**Feature Selection Method:** CORR   
**Dataset:** QUT-DV25 (Opensnoop)   
**Developed By:** eDySec Research Team   
**Plartform:** Ubuntu

All experiments in this notebook were conducted using **Python 3.10** with the following libraries:

`pandas==1.5.3`,  
`scikit-learn==1.2.2`,  
`openpyxl`,  
`numpy==1.23.5`,  
`scipy==1.9.3`,  
`tensorflow==2.11.0`,  
`matplotlib==3.7.1`,  
`seaborn==0.12.2`,  
`joblib==1.3.2`,  
`shap==0.41.0`,  
`lime`,  
`flaml[automl]==2.5.0`,  
`notebook==6.5.6`,  
`pywinpty==2.0.10`  (Only for windows)  `threadpoolctl==3.1.0` (for Ubuntu)   
`terminado==0.17.1`,  
`transformers==4.49.0`.

#### Full Environment Setup: https://github.com/tanzirmehedi/eDySec

These versions were used to ensure **consistent and reproducible experimental results**.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from scipy.sparse import hstack
from scipy.sparse import csr_matrix
from sklearn.metrics import (
    accuracy_score, confusion_matrix, roc_auc_score, roc_curve,
    precision_recall_fscore_support, cohen_kappa_score
)
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
import matplotlib.pyplot as plt
import seaborn as sns
import time
import joblib
import gc
from sklearn.feature_selection import f_classif

In [2]:
# This will allow TensorFlow to allocate as much GPU memory as needed, including full 16GB if needed.
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        # Disable memory growth (optional - default is False)
        tf.config.experimental.set_memory_growth(gpus[0], False)
        print("Memory growth disabled (default behavior).")
    except RuntimeError as e:
        print("Error:", e)

Memory growth disabled (default behavior).


In [3]:
file_path = 'QUT-DV25_'+DATASET_NAME+'_Traces.csv'
data = pd.read_csv(file_path)

In [4]:
gc.collect()
tf.keras.backend.clear_session()


In [5]:
data.head()

,Package_Name,Total_Paths,Total_Error,Total_ File_Descriptor,Python_Related_Keywords,Install_Package_Keywords,Root_DIR_Installation,Temporary_DIR_Installation,Home_DIR_Installation,User_Access,Sys_Access,Etc_DIR_Installation,Other_DIR_Installation,Level
0,10Cent10-999.0.4.tar.gz,8045,8045,29,1083,2305,0,71,792,411,1225,151,5395,1
1,10Cent11-999.0.4.tar.gz,4790,4790,12,1616,843,2,87,1118,721,1200,104,1558,1
2,11Cent-999.0.0.tar.gz,26159,26159,27,3299,4536,0,200,2485,1072,1236,450,20716,1
3,11Cent-999.0.1.tar.gz,11194,11194,28,1521,2558,2,97,1035,891,1211,265,7693,1
4,11Cent-999.0.2.tar.gz,13561,13561,35,2656,5265,501,17,1497,1719,1219,291,8317,1


In [6]:

all_pattern_columns = ['Total_Paths', 'Total_Error', 'Total_ File_Descriptor',
       'Python_Related_Keywords', 'Install_Package_Keywords',
       'Root_DIR_Installation', 'Temporary_DIR_Installation',
       'Home_DIR_Installation', 'User_Access', 'Sys_Access',
       'Etc_DIR_Installation', 'Other_DIR_Installation']


data[all_pattern_columns] = data[all_pattern_columns].fillna(0)

# Target variable
X = data[all_pattern_columns]
y = data['Level']

# 1. Compute correlation with target
correlation_with_target = X.apply(lambda col: col.corr(y) if np.issubdtype(col.dtype, np.number) else 0)

# 2. Drop features highly correlated with each other (|r| >= 0.5)
corr_matrix = X.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

# Features to drop due to high correlation
to_drop = [column for column in upper.columns if any(upper[column] >= 0.5)]

# 3. Keep features with |r| < 0.5
selected_features = [f for f in X.columns if f not in to_drop]

# 4. If you have feature importance scores (IMS) from ML models, apply threshold
# Example: keep features with IMS > 0.05 in at least one model
# Assuming ims_dict = {'feature_name': max_importance_across_models, ...}
# selected_features = [f for f in selected_features if ims_dict.get(f, 0) > 0.05]

# 5. Final selected features DataFrame
X_selected_df = X[selected_features]

print("Selected features:", list(X_selected_df.columns))
print("Number of features kept:", X_selected_df.shape[1])
print("Number of features dropped:", X.shape[1] - X_selected_df.shape[1])



# # Scale features before combining
# scaler = StandardScaler(with_mean=False)  # Sparse matrices need `with_mean=False`
# X_scaled = scaler.fit_transform(X)
# X = pd.DataFrame(X_scaled, columns=all_pattern_columns)



Selected features: ['Total_Paths', 'Total_ File_Descriptor', 'Temporary_DIR_Installation', 'Sys_Access']
Number of features kept: 4
Number of features dropped: 8


In [7]:
selected_features = ['Total_Paths', 'Total_ File_Descriptor', 'Temporary_DIR_Installation', 'Sys_Access']